# Evidence-regularized local surrogates

**MedLIME Learning Track — Notebook 11/12** · Curriculum weeks 29–32 · Suggested time: 60–100 minutes

**Before starting:** Notebooks 04 and 10; ridge regression intuition.

**Learning goal:** Fit a sparse local surrogate with an evidence-derived soft prior and inspect hyperparameters.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## Put evidence in the objective
The toy loss is

\[L(w,b)=\frac1N\sum_i \pi_i\,(f(x_i)-b-z_i^Tw)^2+\lambda_1\lVert w\rVert_1+\lambda_2\lVert w-w_{ref}\rVert_2^2.\]

`lambda1` encourages fewer important patches; `lambda2` encourages agreement with the neighbor evidence. Both can also *hurt* if chosen poorly. This mirrors the paper's broad EBR motivation but is **not asserted to reproduce its exact optimization**.

In [ ]:
query,lesion=synthetic_image(seed=27,lesion=True)
reference=np.stack([synthetic_image(3000+i,True)[0] for i in range(18)])
reference_coef=np.asarray([lime_explain(ref,n=75,seed=3)['coef'] for ref in reference])
prior,ids,weights=evidence_prior(query,reference,reference_coef,k=5)
base=lime_explain(query,n=170,seed=9)
regularized=lime_explain(query,n=170,seed=9,l1=.003,l2=.15,prior=prior)
show_images([query,lesion.astype(float),base['map'],coef_map(prior),regularized['map']],
 ['query','lesion mask','LIME','evidence prior','regularized LIME'],cmap='viridis',ncols=3)
plt.show()

## Diagnose the trade-off
Increase the prior penalty, and the learned patch weights should approach the reference weights. But an inaccurate prior may degrade lesion localization even as the objective gets smaller.

In [ ]:
for strength in [0,.01,.1,.4,2.]:
    r=lime_explain(query,n=170,seed=9,l1=.003,l2=strength,prior=prior)
    print('lambda2=',strength,'local R²=',round(r['r2'],3),
          '||w-prior||=',round(float(np.linalg.norm(r['coef']-prior)),3),
          'AUPRC=',round(auprc(lesion,r['map']),3))

## Why regularize at all?
With limited black-box queries, many patch coefficients can fluctuate. A sparse penalty and a reference-informed penalty trade freedom for constraints. Constraints lower variance when useful, but create bias when wrong.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(9,3))
axs[0].bar(range(16),base['coef']);axs[0].set_title('No evidence penalty')
axs[1].bar(range(16),regularized['coef']);axs[1].set_title('Evidence penalty')
for ax in axs: ax.set_xlabel('Patch id');ax.set_ylabel('Coefficient')
plt.tight_layout();plt.show()

## What you should now understand
- The local fit, sparsity, and evidence penalties are distinct competing objectives.
- Prior quality determines whether regularization helps or hinders.
- Hyperparameters must be selected without tuning against held-out test masks.

## Try it yourself (do not just read the code)
1. Sweep `l1` across 0, 0.002, 0.01, and 0.1; count coefficients whose magnitude exceeds 0.01.
2. Construct an intentionally wrong prior concentrated in a corner and show its effect.

## Exit ticket
**Question:** When could an agreement-with-evidence penalty make a medically important rare lesion *less* visible?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.